<img src="../assets/orchestration_overview.svg" alt="BTP Orchestration 호출 흐름" style="display:block;width:100%;max-width:1200px;margin:0 auto 24px;border-radius:12px;">


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 등록된 Orchestration 선택 (UI)</h2>
<p style="margin: 7px 0;">UI는 <strong>1. 목록 조회 → 2. 상세 불러오기 → 3. 호출</strong> 순서로 안내합니다. 이전 단계가 완료되기 전에는 다음 버튼이 비활성화됩니다.</p>
</div>
</div>

In [1]:
from pathlib import Path
import importlib
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
import src.orchestration.registry_ui as registry_ui
registry_ui = importlib.reload(registry_ui)

orchestration_registry_ui = registry_ui.build_orchestration_registry_ui()
orchestration_registry_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', width='100%', height=900)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">조회와 호출의 경계</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;"><li>목록과 상세 조회는 BTP Registry 상태를 변경하지 않습니다.</li><li>호출은 선택한 Orchestration ID를 사용합니다.</li><li>새 Orchestration 등록과 삭제는 이 노트북의 범위가 아닙니다.</li></ul>
</div>
</div>

<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎 선택 심화</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">공개 core API로 목록 조회하기</h2>
<p style="margin: 7px 0;">아래 셀도 같은 public API를 사용합니다. 필터를 비우면 등록된 전체 목록을 조회하고, 선택한 항목의 <code>id</code>를 호출에 전달합니다.</p>
</div>
</div>

In [2]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').is_dir())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '3_gen_ai_hub_services')]
from src.orchestration.lesson_support.config import load_settings
from src.orchestration.lab_core import get_orchestration_config, input_placeholders, invoke_saved_orchestration_config, list_orchestration_configs
from src.orchestration.str_lab import (
    SYNTHETIC_STR_DATABASE_RESULT, build_str_database_input,
    summarize_str_database_result,
)
settings = load_settings()

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 사례 · STR 초안 작성</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">DB 조회 결과: 입력 → Orchestration → 담당자 검토</h2>
<p style="margin: 7px 0;">아래 데이터는 고객·계좌·원장 테이블을 조회한 것처럼 구성한 교육용 가명 JSON입니다. 출력은 신고서 제출본이 아닌 내부 검토용 초안이며, 사실 확인·신고 여부·최종 내용은 컴플라이언스 담당자가 결정해야 합니다.</p>
</div>
</div>

In [3]:
# INPUT: 가명·마스킹된 DB 조회 결과입니다.
str_database_result = SYNTHETIC_STR_DATABASE_RESULT
str_database_input = build_str_database_input(str_database_result)
{
    'case_id': str_database_result['case_id'],
    'source_tables': ['customer_master', 'account_master', 'ledger_transactions'],
    'ledger_transaction_count': len(str_database_result['ledger_transactions']),
    'orchestration_input': str_database_input,
}


{'case_id': 'STR-TRAIN-2026-001',
 'source_tables': ['customer_master', 'account_master', 'ledger_transactions'],
 'ledger_transaction_count': 4,
 'orchestration_input': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 입력 해석</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 원장 JSON에서 검토 지표 확인하기</h2>
<p style="margin: 7px 0;">공개 core API가 원장 거래 건수·입출금 합계·수취 계좌 수·거래 시간 범위를 계산합니다. 이 값은 모델의 판단을 대체하지 않고, 제공 사실을 검토하는 기준입니다.</p>
</div>
</div>

In [4]:
# INPUT → OUTPUT: 공개 core API가 DB 원장 값을 사실 기반으로 요약합니다.
transaction_summary = summarize_str_database_result(str_database_result)
transaction_summary

{'case_id': 'STR-TRAIN-2026-001',
 'ledger_transaction_count': 4,
 'total_credit_krw': 9800000,
 'total_debit_krw': 9600000,
 'masked_debit_counterparty_count': 3,
 'time_window_minutes': 16.0,
 'timeline': ['2026-09-14T09:12:00+09:00 | CREDIT | 9,800,000 KRW | ***-***-1842',
  '2026-09-14T09:19:00+09:00 | DEBIT | 3,100,000 KRW | ***-***-7711',
  '2026-09-14T09:23:00+09:00 | DEBIT | 3,200,000 KRW | ***-***-5520',
  '2026-09-14T09:28:00+09:00 | DEBIT | 3,300,000 KRW | ***-***-9034']}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 Prompt 구성</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. DB 조회 결과 JSON을 Config 입력으로 사용하기</h2>
<p style="margin: 7px 0;">입력은 원천 JSON을 그대로 유지합니다. STR 초안의 구조·금지 사항·검토 원칙은 Orchestration의 System Prompt에서 제어합니다.</p>
</div>
</div>

In [5]:
# INPUT → OUTPUT: 검증된 DB 조회 결과를 JSON 문자열로 직렬화합니다.
{
    'input_format': 'application/json',
    'serialized_database_result': str_database_input,
    'character_count': len(str_database_input),
}


{'input_format': 'application/json',
 'serialized_database_result': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n  ],\n  "data_limitations": [\n    "모든 식별자는 교육용 가명 또는 마스킹 값입니다.",\n    "상대방 실소유자 정보와 거래 목적 증빙은 본 조회 결과

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. 등록된 Orchestration 목록 조회</h2>
<p style="margin: 7px 0;">필터를 비우면 현재 resource group에서 조회 가능한 모든 Registry Orchestration을 가져옵니다. 결과가 많으면 scenario, name, version을 지정해 좁힐 수 있습니다.</p>
</div>
</div>

In [6]:
registered = list_orchestration_configs(name = 'aml_report_config'
    # scenario='devday', name='assistant', version='1.0.0',  # 필요하면 필터를 사용하세요.
)
registered

{'count': 1,
 'resources': [{'id': '0a0fff53-09cf-4ee2-9af1-0a2a312f507e',
   'name': 'aml_report_config',
   'version': '0.0.1',
   'scenario': 'aml_report',
   'creation_timestamp': '2026-10-21T04:04:05.315000',
   'managed_by': 'imperative',
   'is_version_head': True}]}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 선택한 Config가 받는 입력 확인</h2>
<p style="margin: 7px 0;">기본값은 목록의 첫 항목입니다. 호출할 항목이 다르면 <code>selected_id</code>를 목록에 표시된 ID로 바꾸고, Config가 요구하는 placeholder 이름을 확인하세요.</p>
</div>
</div>

In [7]:
if not registered['resources']:
    raise RuntimeError('등록된 Orchestration이 없습니다. 목록 필터와 resource group을 확인하세요.')

selected_id = registered['resources'][0]['id']  # 목록의 다른 id로 바꿔 선택할 수 있습니다.
selected_config = get_orchestration_config(selected_id)
{
    'config_id': selected_id,
    'input_placeholders': input_placeholders(selected_config),
}


{'config_id': '0a0fff53-09cf-4ee2-9af1-0a2a312f507e',
 'input_placeholders': ['input_var1']}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 Config 계약 확인</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. DB 조회 결과를 Config placeholder에 매핑하기</h2>
<p style="margin: 7px 0;">등록된 Config마다 입력 이름이 <code>question</code> 또는 <code>user_query</code>처럼 다를 수 있습니다. 아래 셀은 호출 전에 전달될 key/value를 보여 주며, 하나의 입력 placeholder가 아닌 Config는 이 실습에서 중단합니다.</p>
</div>
</div>

In [8]:
# INPUT: Registry Config에서 추출한 placeholder 이름 + DB 조회 결과 JSON
str_placeholder_names = input_placeholders(selected_config)
if len(str_placeholder_names) != 1:
    raise RuntimeError(f'이 STR 실습은 입력 placeholder가 정확히 하나인 Config가 필요합니다. 현재 값: {str_placeholder_names}')

# OUTPUT: SDK run()에 전달될 placeholder_values 미리 보기
str_placeholder_values = {str_placeholder_names[0]: str_database_input}
{
    'config_id': selected_id,
    'placeholder_values': str_placeholder_values,
}


{'config_id': '0a0fff53-09cf-4ee2-9af1-0a2a312f507e',
 'placeholder_values': {'input_var1': '{\n  "account_master": [\n    {\n      "account_no_masked": "110-***-0427",\n      "account_status": "ACTIVE",\n      "currency": "KRW",\n      "customer_id": "CUST-0427",\n      "opened_at": "2023-04-18"\n    }\n  ],\n  "case_id": "STR-TRAIN-2026-001",\n  "customer_master": [\n    {\n      "business_type": "온라인 유통업",\n      "customer_id": "CUST-0427",\n      "customer_name_masked": "김*준",\n      "customer_type": "INDIVIDUAL_BUSINESS",\n      "kyc_last_verified_at": "2026-03-02",\n      "onboarded_at": "2023-04-18",\n      "risk_grade": "MEDIUM"\n    }\n  ],\n  "customer_transaction_summary": [\n    {\n      "avg_daily_credit_krw": 1210000,\n      "avg_daily_debit_krw": 1180000,\n      "customer_id": "CUST-0427",\n      "max_single_credit_krw": 2400000,\n      "period_days": 90,\n      "transaction_count": 86\n    }\n  ],\n  "data_limitations": [\n    "모든 식별자는 교육용 가명 또는 마스킹 값입니다.",\n    "상대방 실소

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 개념 · 실습 단계</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">6. 선택한 Orchestration 호출</h2>
<p style="margin: 7px 0;">교육용 DB 조회 결과 JSON을 입력으로 전달합니다. 선택한 Config가 하나의 입력 placeholder를 지원하면 함수가 그 이름에 맞춰 요청을 구성합니다. 출력은 자동 생성된 내부 검토용 초안으로만 취급하세요.</p>
</div>
</div>

In [9]:
# CALL: 위에서 확인한 key/value를 명시적으로 전달합니다.
str_draft_result = invoke_saved_orchestration_config(
    None, settings, config_id=selected_id, placeholder_values=str_placeholder_values
)

# OUTPUT: 제출용 보고서가 아닌 담당자 검토용 초안입니다.
str_draft_result

{'config_id': '0a0fff53-09cf-4ee2-9af1-0a2a312f507e',
 'answer': '# 내부 검토용 STR 초안  \n**사건번호: STR-TRAIN-2026-001**\n\n## 1. 의심거래 개요\n\n- **대상 고객**: 김*준, 개인사업자, 온라인 유통업  \n- **고객번호**: CUST-0427  \n- **계좌**: 110-***-0427, 원화, 정상상태  \n- **고객 위험등급**: 중위험  \n- **계좌 개설일**: 2023-04-18  \n- **최근 KYC 확인일**: 2026-03-02  \n- **분석 대상 거래일**: 2026-09-14  \n- **주요 의심거래 규모**: 총 입금 9.80백만원, 총 출금 9.60백만원, 순증가 0.20백만원  \n- **주요 의심 징후**:  \n  - 9.80백만원의 단일 입금 직후 16분 이내 3건, 합계 9.60백만원이 출금됨  \n  - 입금액의 약 98.0%가 단시간 내 서로 다른 3개 계좌로 분산 출금됨  \n  - 출금 상대방은 은행코드 020, 081, 088의 계좌로 확인됨  \n  - 해당 단일 입금액은 최근 90일 평균 일일 입금액 1.21백만원의 약 8.1배이며, 최근 90일 최대 단일 입금액 2.40백만원의 약 4.1배에 해당함  \n  - 거래 목적, 상대방과의 관계 및 실소유자 정보에 대한 증빙이 확인되지 않음  \n\n## 2. 주요 거래내역 및 자금 흐름\n\n2026-09-14 09:12, 계좌 110-***-0427로 상대방 계좌 ***-***-1842로부터 **9.80백만원**이 입금됨. 입금 직후 계좌 잔액은 **9.8345백만원**으로 확인됨.\n\n이후 다음과 같이 단시간 내 출금이 발생함.\n\n| 일시 | 거래구분 | 금액 | 상대방 계좌 | 은행코드 | 거래 후 잔액 |\n|---|---:|---:|---|---:|---:|\n| 2026-09-14 09:19 | 출금 | 3.10백만원 | ***-***-7711

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘 출력 검토</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">7. 생성된 초안을 사람 검토 기준으로 점검하기</h2>
<p style="margin: 7px 0;">키워드 검사는 품질 판정이나 컴플라이언스 승인을 자동화하지 않습니다. 담당자가 확인할 최소 항목을 빠르게 표시하는 보조 도구로만 사용합니다.</p>
</div>
</div>

In [10]:
# INPUT: Orchestration이 반환한 answer 문자열
str_draft_text = str_draft_result['answer']
required_sections = ['사건 개요', '거래 사실', '의심 사유', '추가 확인 필요 사항', '담당자 검토 의견']

# OUTPUT: 사람 검토를 위한 체크리스트입니다. False면 초안을 수정하거나 담당자가 보완합니다.
str_review_check = {
    'is_internal_draft_notice_present': '내부 검토용 초안' in str_draft_text,
    'section_presence': {section: section in str_draft_text for section in required_sections},
    'contains_unmasked_example_account': '123-456' in str_draft_text,
    'manual_review_required': True,
}
str_review_check

{'is_internal_draft_notice_present': False,
 'section_presence': {'사건 개요': False,
  '거래 사실': False,
  '의심 사유': False,
  '추가 확인 필요 사항': False,
  '담당자 검토 의견': False},
 'contains_unmasked_example_account': False,
 'manual_review_required': True}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
<div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">🧪 실패 케이스</div>
<div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">8. 필수 거래 데이터가 없을 때 호출을 막기</h2>
<p style="margin: 7px 0;">외부 서비스를 호출하기 전에 사례의 최소 요건을 검증합니다. 아래 셀에서 빈 거래 목록을 넣어 예상 오류를 직접 확인하세요.</p>
</div>
</div>

In [11]:
from copy import deepcopy

# INPUT: 의도적으로 비운 DB 원장 결과
invalid_str_database_result = deepcopy(SYNTHETIC_STR_DATABASE_RESULT)
invalid_str_database_result['ledger_transactions'] = []

# EXPECTED ERROR: 모델 호출 전 입력 검증에서 중단됩니다.
validation_result = None
try:
    build_str_database_input(invalid_str_database_result)
except ValueError as error:
    validation_result = {'error_type': type(error).__name__, 'message': str(error)}
validation_result


{'error_type': 'ValueError',
 'message': 'DB 조회 결과에는 한 건 이상의 ledger_transactions가 필요합니다.'}